# 创建一个 Llava 模型

本 notebook 将基于 clip-vit-large-patch-14-336 和 Qwen1.5-4b-Chat 模型，搭建 Llava-1.5 架构。

由于 torch 和代码的兼容性问题，后面的部分（包括数据准备和训练）我们将降级 transformers 回到原始仓库相近的版本。前面一个仓库是针对于现在的 HF 文档的适配与解读，所以使用更高级的 transformers 理解如何调用 LLaVA。

In [1]:
# pip install transformers==4.41.2 huggingface_hub modelscope

## 下载模型

和 [使用Llava-1.5-7b.ipynb](使用Llava-1.5-7b.ipynb) 中一样，我们先下载好模型。如果镜像站下载太慢，建议使用魔塔社区下载。

```bash

pip install -U huggingface_hub

export HF_ENDPOINT=https://hf-mirror.com

huggingface-cli download --resume-download openai/clip-vit-large-patch14-336 --local-dir openai/clip-vit-large-patch14-336 --local-dir-use-symlinks False


huggingface-cli download --resume-download Qwen/Qwen1.5-4B-Chat --local-dir Qwen1.5-4B-Chat --local-dir-use-symlinks False
```

### 使用魔塔社区下载

注意下载代码只能运行一次！！！否则需要重新改 Qwen Tokenizer 的 json 文件。

In [2]:
use_modelscope=True
download=False      # 只运行一次，确认下载好了就设置为 False

In [3]:
if use_modelscope and download:
    # 安装或更新 modelscope 库（如果环境中没有的话）
    # pip install -U modelscope
    
    # 从魔搭下载模型
    from modelscope import snapshot_download
    
    model_dir = snapshot_download(
        'AI-ModelScope/clip-vit-large-patch14-336',       # 魔搭上的模型 ID，与你给的链接一致
        cache_dir='/root/autodl-tmp',      # 强烈建议指向数据盘，避免系统盘被占满
        revision='master'                  # 主分支，通常就是最新的稳定版
    )
    
    print(f"vision 模型已下载到: {model_dir}")
    
    model_dir = snapshot_download(
        'Qwen/Qwen1.5-4B-Chat',       # 魔搭上的模型 ID，与你给的链接一致
        cache_dir='/root/autodl-tmp',      # 强烈建议指向数据盘，避免系统盘被占满
        revision='master'                  # 主分支，通常就是最新的稳定版
    )
        
    print(f"LLM 模型已下载到: {model_dir}")

## 修改 Qwen tokenizer

原始的 Qwen Tokenizer 中是不包含 Llava 中引入的 `<image>` 词元的，如果直接使用的话会被拆成三个 token：`<`、`image`、`>`。为了让 Qwen 能够识别我们引入的多模态 `<image>` 词元，需要我们手动添加到下载的文件中的`tokenizer_config.json`中：

1. 在`added_tokens_decoder`里面，加上：
   ```json
    "151646": {
        "content": "<image>",
        "lstrip": false,
        "normalized": false,
        "rstrip": false,
        "single_word": false,
        "special": true
        }
    ```
2. 在`additional_special_tokens` 里面加上 `"<image>"`
   
添加之后简单做一下测试，注意替换模型路径为上面的下载代码打印出来的 LLM 路径。

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 请替换路径
modify_qwen_tokenizer_dir = "/root/autodl-tmp/models/Qwen--Qwen1.5-4B-Chat/snapshots/master"
modify_qwen_tokenizer = AutoTokenizer.from_pretrained(modify_qwen_tokenizer_dir)

modify_qwen_tokenizer.encode("<image>")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


[151646]

### 加了这个新 token，需要修改模型的 embedding 模块么？

不需要。这是使用 Qwen 模型的好处。Qwen 在训练时给 embedding 层预留了较大的空间，我们添加一些特殊词元不会导致报错。

In [5]:
qwen_model = AutoModelForCausalLM.from_pretrained(modify_qwen_tokenizer_dir, device_map='cuda:0', 
                                                  torch_dtype=torch.bfloat16)

# 打印添加 <image> 后 Qwen tokenizer 的长度
print(len(modify_qwen_tokenizer))

# 打印 Qwen Embedding 维度
print((qwen_model.model.embed_tokens,))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

151647
(Embedding(151936, 2560),)


可以看到 embed_tokens 的长度比 tokenizer 的长度要大。

## Llava 模型初始化

根据上面的下载代码打印的路径，配置 Vision Model 和 Language Model 的路径。

In [6]:
# 请修改成你的路径
clip_model_name_or_path = (
    "/root/autodl-tmp/models/AI-ModelScope--clip-vit-large-patch14-336/snapshots/master"
)
qwen_model_name_or_path = "/root/autodl-tmp/models/Qwen--Qwen1.5-4B-Chat/snapshots/master"

使用 transformers 接口，将两个模型读入内存，同时提前加载 tokenizer（后面构造 LlavaConfig 需要它的 pad/eos）。（这里直接读入显存，如果 OOM 把 device 改回 cpu）

In [7]:
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

clip_model = AutoModel.from_pretrained(
    clip_model_name_or_path,
    device_map="cuda:0",
)
llm_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name_or_path,
    device_map="cuda:0",
    torch_dtype=torch.bfloat16,
)

# 提前加载 tokenizer：Qwen 的 model config 没有 pad_token_id，
# 需要在构造 LlavaConfig 之前从 tokenizer 拿到 pad/eos。
llm_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name_or_path)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


使用内存中的模型，初始化 llava 模型的配置，并创建一个 llava 模型对象。

In [8]:
from transformers import LlavaForConditionalGeneration, LlavaConfig

# 初始化 CLIP-vision config
vision_config = clip_model.vision_model.config

# 初始化 LLM config
text_config = llm_model.config

# 用上面的 config 初始化 llava-1.5-7b style configuration
configuration = LlavaConfig(vision_config, text_config)

# LlavaConfig 不会自动设置 pad/eos；Qwen 的 model config 里也没有 pad_token_id。
# 在构造模型之前写入，LlavaForConditionalGeneration 才能把 self.pad_token_id
# 和 self.generation_config 一起初始化正确。
configuration.pad_token_id = llm_tokenizer.pad_token_id   # 151643
configuration.eos_token_id = llm_tokenizer.eos_token_id   # 151645

# 使用 llava config 创建 llava 模型
model = LlavaForConditionalGeneration(configuration)

作为检验，我们看看 Llava 中的 embedding 层和 CLIP 中的 embedding 层，检查形状信息是否成功导入 Llava 配置。

In [9]:
model.vision_tower.vision_model.embeddings

CLIPVisionEmbeddings(
  (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
  (position_embedding): Embedding(577, 1024)
)

In [10]:
clip_model.vision_model.embeddings

CLIPVisionEmbeddings(
  (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
  (position_embedding): Embedding(577, 1024)
)

将权重复制到 llava 模型中。

In [11]:
# 更换视觉模型
model.vision_tower.vision_model = clip_model.vision_model

# 更换语言模型
model.language_model = llm_model

作为检验，我们看看 Qwen 的 embedding 权重和 Llama 的 embedding 权重，检查是否赋值成功。

In [12]:
llm_model.model.embed_tokens.weight.data[:, :2]

tensor([[ 4.7302e-03, -6.2866e-03],
        [-5.9814e-03,  1.1841e-02],
        [-5.8899e-03,  1.5747e-02],
        ...,
        [-3.3379e-05,  8.5831e-06],
        [-3.2425e-05,  7.1824e-06],
        [-3.1948e-05, -2.9206e-06]], device='cuda:0', dtype=torch.bfloat16)

In [13]:
model.language_model.model.embed_tokens.weight.data[:, :2]

tensor([[ 4.7302e-03, -6.2866e-03],
        [-5.9814e-03,  1.1841e-02],
        [-5.8899e-03,  1.5747e-02],
        ...,
        [-3.3379e-05,  8.5831e-06],
        [-3.2425e-05,  7.1824e-06],
        [-3.1948e-05, -2.9206e-06]], device='cuda:0', dtype=torch.bfloat16)

在 Llava 模型中还有一些属性需要配置，因为 `LlavaConfig(vision_config, text_config)` 不会自动把特殊 token 设好。这里主要关注 `pad_token_id`、`eos_token_id` 和 `image_token_index`。

需要注意两点：

1. Qwen1.5-4B-Chat 的 `config.json` 里**没有** `pad_token_id`（只有 `bos_token_id=151643`、`eos_token_id=151645`），所以 `llm_model.config.pad_token_id` 是 `None`。pad 应该从 tokenizer 取（151643）。
2. `LlavaForConditionalGeneration.__init__` 会把 `config.pad_token_id` 缓存成 `model.pad_token_id`，同时用 `GenerationConfig.from_model_config(config)` 生成独立的 `model.generation_config`。**构造完成后再改 `model.config` 不会同步这两个副本**。

因此最稳妥的做法是在构造模型之前就把这两个值写进 `configuration`（见上一节）；如果构造之后再改，就必须三处一起改。

`image_token_index` 是个例外：`forward` 每次都从 `self.config.image_token_index` 读取，所以可以在 tokenizer 加载后再设置。

In [14]:
# 构造完成后检查三处是否一致：
# config 会被 save_pretrained 写进 config.json；
# model.pad_token_id 和 generation_config 是构造时从 config 派生出来的副本。
print("config:", model.config.pad_token_id, model.config.eos_token_id)
print("model.pad_token_id:", model.pad_token_id)
print("generation_config:", model.generation_config.pad_token_id, model.generation_config.eos_token_id)

# image_token_index 在下一步 tokenizer 初始化后设置；
# forward 每次都从 config 读取，所以后设是有效的。

config: 151643 151645
model.pad_token_id: 151643
generation_config: 151643 151645


## Llava Tokenizer 初始化

我们用 Qwen 替换了原始模型的 Vicuna，所以需要同步修改 tokenizer。这一步的目标是：把已经改好的 Qwen tokenizer 和 CLIP 的 image processor 组装成 `LlavaProcessor`。

Qwen Tokenizer 已经在上文加载。这里先确认 `<image>` 已经被注册成单个特殊 token。

In [15]:
# llm_tokenizer 已在上文加载，这里只验证 <image> 是否被识别成单个 token
image_token_id = llm_tokenizer.encode("<image>")[0]  # 取 [0] 因为 encode 默认返回 list
image_token_id
# 应该输出 151646，而不是被拆成多个 token

151646

将特殊 token id 加入 Llava 模型的配置文件中。

In [16]:
model.config.image_token_index = image_token_id

对于 CLIP 的 processor，与参考仓库不同，这里不需要再把 CLIP 的 `preprocessor_config.json` 手动复制到模型目录。

关于在 [使用Llava-1.5-7b.ipynb](使用Llava-1.5-7b.ipynb) 中提到的 `<image>` 的展开位置问题。在 transformers-4.41.2 中，`LlavaProcessor` 只是一个简单封装，实际上会将文本交给 tokenizer，图像交给 `CLIPImageProcessor`，`<image>` 的展开不在 processor 里。真正的展开发生在 `LlavaForConditionalGeneration.forward` 的 `_merge_input_ids_with_image_features` 中：模型用 CLIP 提取 patch 特征，再按 `image_token_index` 把输入 embedding 里的 `<image>` 替换成 576 个 visual token，并同步扩展 `attention_mask` 和 `labels`。

因此这里**不需要**设置 `patch_size`、`vision_feature_select_strategy`、`num_additional_image_tokens`。这些字段是 4.45+（例如 4.57.6）的 processor 负责展开 `<image>` 时才需要的。

In [17]:
from transformers import LlavaProcessor, CLIPImageProcessor

image_processor = CLIPImageProcessor.from_pretrained(clip_model_name_or_path)

# 4.41.2 的 LlavaProcessor 只接收 image_processor 和 tokenizer，
# 它不负责展开 <image>，所以不需要 patch_size 等元数据。
llava_processor = LlavaProcessor(
    image_processor=image_processor,
    tokenizer=llm_tokenizer,
)

保存模型 + Tokenizer + Processor 到同一个目录。

In [18]:
save_dir = "show_model/model001"

model.save_pretrained(save_dir)
# LlavaProcessor 会同时保存 tokenizer 和 image processor。
# 4.41.2 的 LlavaProcessor 没有额外属性，save_pretrained 返回 []，不会生成
# processor_config.json；preprocessor_config.json 由 image processor 自己写出。
llm_tokenizer.save_pretrained(save_dir)
llava_processor.save_pretrained(save_dir)

[]

保存后目录里应该同时包含：

- `config.json`（LlavaConfig，含 `image_token_index`、`pad_token_id`、`eos_token_id`）
- `model.safetensors` 或分片权重
- `tokenizer_config.json`（含 `<image>` 注册）
- `preprocessor_config.json`（CLIPImageProcessor 的配置；`processor_class` 会被标成 `"LlavaProcessor"`，方便 `AutoProcessor` 识别）

## 测试效果

在 notebook 中**重启内核后直接运行**下面的代码（跳过上方的代码块），测试之前保存的模型能不能直接使用。

In [1]:
from transformers import LlavaProcessor, LlavaForConditionalGeneration
import torch

model_name_or_path = "show_model/model001"

llava_processor = LlavaProcessor.from_pretrained(model_name_or_path)
model = LlavaForConditionalGeneration.from_pretrained(
    model_name_or_path, device_map="cuda:0", torch_dtype=torch.bfloat16
)
llm_tokenizer = llava_processor.tokenizer

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

进行一次前向推理。注意此时连接视觉模型和语言模型的 MLP 权重是默认初始化的权重，我们将在后续的训练中让网络学习这部分权重。这里我们采用有 system 字段的模格式，因此需要手动在 messages 中指定（原因在 [使用Llava-1.5-7b.ipynb](使用Llava-1.5-7b.ipynb) 中）。

In [2]:
from PIL import Image

# Qwen 接受这种格式的 messages
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "<image>\nWhat is in this image?"
    }
]
image_path = "000000039769.jpg"
image = Image.open(image_path)

# 直接使用底层的 tokenizer 来应用 chat template
prompt = llm_tokenizer.apply_chat_template(
    messages,
    tokenize=False,           # 返回字符串，而不是 token ids
    add_generation_prompt=True
)
inputs = llava_processor(text=prompt, images=image, return_tensors="pt").to(
    model.device, dtype=torch.bfloat16
)

generate_ids = model.generate(**inputs, max_new_tokens=20)
# 这里保留特殊 token，用于观察输出
gen_text = llava_processor.batch_decode(
    generate_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False
)[0]    # 取 0 因为 batch_decode 支持批量解码，第一个维度是 batch 维

print(gen_text)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<image>
What is in this image?<|im_end|>
<|im_start|>assistant
The image shows a person wearing a black hoodie, jeans, and sneakers, standing in front of a


可以打印输入 prompt 看看 Qwen 的输入格式。

In [3]:
print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<image>
What is in this image?<|im_end|>
<|im_start|>assistant

